In [13]:
# -*- coding: utf-8 -*-
# 【CP2-14 重试策略】RetryPolicy 对瞬时异常的自动重放
# 文件：CP2/14_retry_ipynb.ipynb
# 作用：本 Cell 演示 【CP2-14 重试策略】RetryPolicy 对瞬时异常的自动重放 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

from debugpy import trace_this_thread
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from typing import TypedDict
from urllib.error import HTTPError

from langgraph.types import RetryPolicy  # 重试策略：定义可重试异常与最大次数
from loguru import logger
from sympy import false


class EmptyState(TypedDict):
    pass

def node_a(state : EmptyState) -> EmptyState:
    logger.info("node_a")
    raise HTTPError(None, 400, "node_a error", None, None)
#               url  code     msg         hdrs   fp

builder = StateGraph(state_schema=EmptyState)  # 创建状态图构建器：绑定状态 Schema
builder.add_node(  # 注册节点到图中
    "node_a",
    node_a,
    retry=RetryPolicy(  # 重试策略：仅对指定异常生效
        # max_attempts=3：最多尝试 3 次（含首次）
        # retry_on=(HTTPError,)：仅 HTTPError 触发重试，其他异常直接抛出
        # jitter=False：关闭随机抖动，固定间隔（指数退避仍生效）
  # 重试策略：定义可重试异常与最大次数
        max_attempts=3,
        jitter=False,  # 若为 True，会在指数退避上加随机抖动以错峰
        retry_on=(HTTPError,)
    )
)

builder.add_edge(START,"node_a")  # 起点扇出
builder.add_edge("node_a",END)  # 汇入终点

graph = builder.compile()  # 编译图：蓝图→可执行对象
try:
    graph.invoke({})  # 触发图执行：传入初始 State + config
except HTTPError as e:
    logger.info(f"node_a error: {e}")


2026-08-06 01:48:37.520 | INFO     | __main__:node_a:16 - node_a
2026-08-06 01:48:38.021 | INFO     | __main__:node_a:16 - node_a
2026-08-06 01:48:39.023 | INFO     | __main__:node_a:16 - node_a
2026-08-06 01:48:39.024 | INFO     | __main__:<module>:38 - node_a error: HTTP Error 400: node_a error
